# Deploying NVIDIA Nemotron-3-Nano-Omni with vLLM

This notebook will walk you through how to run the `nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning` model with vLLM.

[vLLM](https://docs.vllm.ai) is a fast and easy-to-use library for LLM inference and serving. 

For more details on the model [click here](https://huggingface.co/nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16)

Prerequisites:
- NVIDIA GPU with recent drivers (≥ 64 GB VRAM for BF16, ≥ 32 GB for FP8, ≥ 20 GB for NVFP4) and CUDA 13.0+
- Python 3.10+

#### Launch on NVIDIA Brev
You can simplify the environment setup by using [NVIDIA Brev](https://developer.nvidia.com/brev). Click the button to launch this project on a Brev instance with the necessary dependencies pre-configured.

Once deployed, click on the "Open Notebook" button to get started with this guide. 

**For H100 (BF16/FP8 models):**

[![Launch on Brev](https://brev-assets.s3.us-west-1.amazonaws.com/nv-lb-dark.svg)](https://brev.nvidia.com/launchable/deploy?launchableID=env-3Cm2gB9j5ROkCbiNKH5SQhERqBV)

**For RTX PRO 6000 (NVFP4 model - requires Blackwell architecture):**

[![Launch on Brev](https://brev-assets.s3.us-west-1.amazonaws.com/nv-lb-dark.svg)](https://brev.nvidia.com/launchable/deploy?launchableID=env-3Cm2uEmMpGCh42xBuU8wdoXD2f9)

## Environment setup

### Verify GPU driver

vLLM 0.20.0 requires an NVIDIA driver that supports **CUDA 13.0**. Check your driver version:

```shell
nvidia-smi | head -5
```

If the **CUDA Version** shown is below 13.0, update the driver:

```shell
# Remove the old driver (adjust 550 to match your installed version)
sudo apt remove -y --purge 'nvidia-*-550'
sudo apt autoremove -y

# Install a CUDA 13.0-compatible driver
sudo apt update
sudo apt install -y nvidia-driver-580
sudo reboot
```

### Install dependencies

In [1]:
#If pip not found
!python -m ensurepip --default-pip

Looking in links: /tmp/tmpvy6ajpu0
Processing /tmp/tmpvy6ajpu0/pip-25.0.1-py3-none-any.whl


In [ ]:
%pip install vllm[audio]==0.20.0 torch nvidia-cutlass-dsl flashinfer-cubin==0.6.8.post1 flashinfer-python==0.6.8.post1

### Additional Setup only for the NVFP4 Model (Blackwell GPUs):

1. Install CUDA Toolkit and C++ compiler
```shell
sudo apt update
sudo apt install -y cuda-toolkit-13-0 g++-12 gcc-12 build-essential
```

2. Set environment variables
```shell
export CUDA_HOME=/usr/local/cuda-13.0
export PATH=$CUDA_HOME/bin:$PATH
```

## Verify GPU

Confirm CUDA is available and your GPU is visible to PyTorch.


In [3]:
# GPU environment check
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Num GPUs: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU[{i}]: {torch.cuda.get_device_name(i)}")

CUDA available: True
Num GPUs: 1
GPU[0]: NVIDIA H100 PCIe


## Load the model

Initialize the Nemotron model in vLLM with BF16 or FP8 (instructions for NVFP4 in the next cell).

In [17]:
from vllm import LLM

llm = LLM(
    model="nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16",
    # Alternative: Load the FP8 quantized version for faster inference and lower memory usage
    # model="nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-FP8",
    trust_remote_code=True,
    dtype="auto",
    max_model_len=131072,
    max_num_seqs=512,
)

print("Model ready")

INFO 04-27 21:42:50 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 131072, 'max_num_seqs': 512, 'disable_log_stats': True, 'model': 'nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16'}
INFO 04-27 21:42:51 [model.py:555] Resolved architecture: NemotronH_Nano_Omni_Reasoning_V3
INFO 04-27 21:42:51 [model.py:1680] Using max model len 131072
INFO 04-27 21:42:51 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
(EngineCore pid=14575) INFO 04-27 21:42:53 [core.py:109] Initializing a V1 LLM engine (v0.20.0) with config: model='nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16', speculative_config=None, tokenizer='nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=131072, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, 

Loading safetensors checkpoint shards:   0% Completed | 0/17 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  47% Completed | 8/17 [00:00<00:00, 76.72it/s]
Loading safetensors checkpoint shards: 100% Completed | 17/17 [00:00<00:00, 104.10it/s]
(EngineCore pid=14575) 


(EngineCore pid=14575) INFO 04-27 21:43:05 [default_loader.py:384] Loading weights took 8.92 seconds
(EngineCore pid=14575) INFO 04-27 21:43:05 [unquantized.py:343] Using MoEPrepareAndFinalizeNoDPEPModular
(EngineCore pid=14575) INFO 04-27 21:43:06 [gpu_model_runner.py:4879] Model loading took 61.57 GiB memory and 10.845026 seconds
(EngineCore pid=14575) INFO 04-27 21:43:06 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
(EngineCore pid=14575) INFO 04-27 21:43:06 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
(EngineCore pid=14575) INFO 04-27 21:43:06 [interface.py:606] Setting attention block size to 1072 tokens to ensure that attention page size is >= mamba page size.
(EngineCore pid=14575) INFO 04-27 21:43:06 [interface.py:630] Padding mamba page size by 1.13% to ensure that mamba page size and attention page size are exactly equal.
(EngineCore pid=14575) INFO

(EngineCore pid=14575) 2026-04-27 21:43:44,491 - INFO - autotuner.py:457 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
[AutoTuner]: Tuning trtllm::fused_moe::gemm2: 100%|██████████| 10/10 [00:07<00:00,  1.32profile/s]
(EngineCore pid=14575) 2026-04-27 21:43:56,336 - INFO - autotuner.py:466 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:03<00:00, 15.43it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 51/51 [00:03<00:00, 14.13it/s]


(EngineCore pid=14575) INFO 04-27 21:44:04 [gpu_model_runner.py:6133] Graph capturing finished in 8 secs, took 0.85 GiB
(EngineCore pid=14575) INFO 04-27 21:44:04 [gpu_worker.py:599] CUDA graph pool memory: 0.85 GiB (actual), 0.64 GiB (estimated), difference: 0.21 GiB (24.2%).
(EngineCore pid=14575) INFO 04-27 21:44:04 [core.py:299] init engine (profile, create kv cache, warmup model) took 57.85 s (compilation: 7.94 s)
(EngineCore pid=14575) INFO 04-27 21:44:04 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
Model ready


Load the NVFP4 quantized version:

In [ ]:
import os
from vllm import LLM

llm = LLM(
    model="nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-NVFP4",
    tensor_parallel_size=1,
    max_model_len=131072,
    kv_cache_dtype="fp8",
    trust_remote_code=True,
    dtype="auto",
    max_num_seqs=8,
    gpu_memory_utilization=0.85
)

print("Model ready")

## Generate responses

Generate text with vLLM using single, batched, and simple streaming examples.

### Single or batch prompts

Send one prompt or a list to run batched generation.

In [18]:
from vllm import SamplingParams

params = SamplingParams(temperature=0.6, max_tokens=200)

# Single prompt
single = llm.generate(["Give me 3 bullet points about vLLM."], sampling_params=params)
print(single[0].outputs[0].text)

# Batch prompts
prompts = [
    "Hello, my name is",
    "The capital of France is",
    "Explain quantum computing in simple terms:"
]
outputs = llm.generate(prompts, sampling_params=params)
for i, out in enumerate(outputs):
    print(f"\nPrompt {i+1}: {out.prompt!r}")
    print(out.outputs[0].text)

Processed prompts: 100%|██████████| 1/1 [00:47<00:00, 47.50s/it, est. speed input: 0.23 toks/s, output: 3.54 toks/s]


 vLLM is an open-source library for high-performance LLM inference. It uses PagedAttention, speculative decoding, and continuous batching to achieve high throughput and low latency. So I need three bullet points summarizing that.

Make concise bullet points. Ensure each bullet is separate. Probably:

- High-performance LLM inference with PagedAttention for efficient memory management.
- Uses speculative decoding to boost throughput and reduce latency.
- Supports continuous batching for dynamic batch sizes and improved latency.

That's three.
</think>
- Leverages **PagedAttention** for efficient memory allocation and high‑throughput LLM inference.  
- Employs **speculative decoding** to dramatically increase throughput and lower latency.  
- Supports **continuous batching**, enabling dynamic batch sizes and consistently low latency.


Processed prompts: 100%|██████████| 3/3 [00:01<00:00,  2.02it/s, est. speed input: 12.12 toks/s, output: 271.41 toks/s]


Prompt 1: 'Hello, my name is'
 Sam. I'm 29, living in New York City and I work in marketing. I'm from Texas, but I've lived in NYC for the last 5 years. I currently live in Manhattan, but I'm thinking about moving to Brooklyn. What are some pros and cons of moving to Brooklyn from Manhattan? I'd like to know some pros and cons of the neighborhoods in Brooklyn, like Williamsburg, DUMBO, and Bushwick. Also, I'm curious about the commute to Manhattan from those areas. Thanks!"

We need to give pros and cons of moving from Manhattan to Brooklyn, focusing on specific neighborhoods: Williamsburg, DUMBO, Bushwick. Also discuss commute to Manhattan. Should be friendly, concise but thorough. Possibly bullet points. Also mention cost of living, vibe, amenities, transport, etc. Provide commute times (subway, ferry). Also maybe include other considerations like lifestyle, food, arts, etc. Use a friendly

Prompt 2: 'The capital of France is'
 Paris.

Prompt 3: 'Explain quantum computing in simple 

### Streamed generation

Print characters as they are produced.

In [7]:
def stream_like(prompt: str, llm: LLM, sampling_params: SamplingParams) -> None:
    outputs = llm.generate([prompt], sampling_params=sampling_params)
    text = outputs[0].outputs[0].text
    print("Response:", end=" ")
    for ch in text:
        print(ch, end="", flush=True)
    print()

stream_like("Write a haiku about GPUs.", llm, SamplingParams(temperature=0.7, max_tokens=512))


Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  2.97it/s, est. speed input: 23.82 toks/s, output: 163.79 toks/s]

Response:  Make sure to include the word "cache" in it.

Here's a possible haiku:

**GPUs accelerate,**  
**Massive parallel computations,**  
**Data cached in fast cache.**  

Let me know if you'd like a revised version!


## OpenAI-compatible server

Serve the model via an OpenAI-compatible API using vLLM.

### Release GPU memory before server mode

This notebook used GPU memory for in-process inference.
Before starting `vllm serve`, run the cleanup cell below to free memory.

If server startup still runs out of memory, restart the kernel, then continue from this section.

In [8]:
import gc
import torch

if "llm" in globals():
    del llm

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Cleanup complete. If server startup still OOMs, restart the kernel before proceeding.")

[rank0]:[W420 17:49:56.657639691 ProcessGroupNCCL.cpp:1524] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


Cleanup complete. If server startup still OOMs, restart the kernel before proceeding.


### Prepare terminal environment

Use the same virtual environment where dependencies were installed.

For example, on Brev run:
```shell
source /home/shadeform/.venv/bin/activate
```

If you are not on Brev, replace this with your own virtual environment path.

### Start server

After cleanup (or after restart if needed), run this in a terminal for BF16/FP8.

Instructions for NVFP4 are in the next cell.

> **FP8:** Replace `nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16` with `nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-FP8` in the command below.

```shell
vllm serve nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16 \
    --served-model-name nemotron \
    --trust-remote-code \
    --dtype auto \
    --host 0.0.0.0 \
    --port 5000 \
    --tensor-parallel-size 1 \
    --max-model-len 131072 \
    --media-io-kwargs '{"video":{"num_frames":512,"fps":1}}' \
    --video-pruning-rate 0.5 \
    --enable-auto-tool-choice \
    --tool-call-parser qwen3_coder \
    --reasoning-parser nemotron_v3
```

For NVFP4, run this in a terminal:

```shell
vllm serve nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-NVFP4 \
    --served-model-name nemotron \
    --trust-remote-code \
    --host 0.0.0.0 \
    --port 5000 \
    --tensor-parallel-size 1 \
    --max-model-len 131072 \
    --max-num-seqs 8 \
    --kv-cache-dtype fp8 \
    --moe-backend triton \
    --media-io-kwargs '{"video":{"num_frames":512,"fps":1}}' \
    --video-pruning-rate 0.5 \
    --enable-auto-tool-choice \
    --tool-call-parser qwen3_coder \
    --reasoning-parser nemotron_v3
```

Your server is now running!

### Use the API

Send chat and streaming requests to your vLLM server using the OpenAI-compatible client.

Note: The model supports two modes - Reasoning ON (default) vs OFF. This can be toggled by setting enable_thinking to False, as shown below.

In [1]:
# Client: Standard chat and streaming
from openai import OpenAI

client = OpenAI(base_url="http://127.0.0.1:5000/v1", api_key="null")

In [ ]:
# Reasoning on (default)
print("Reasoning on")
resp = client.chat.completions.create(
    model="nemotron",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1,
    max_tokens=1024,
)

print("Reasoning:", resp.choices[0].message.reasoning, "\nContent:", resp.choices[0].message.content)
print("\n")

# Reasoning off
print("Reasoning off")
resp2 = client.chat.completions.create(
    model="nemotron",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Give me 3 interesting facts about vLLM."}
    ],
    temperature=0,
    max_tokens=256,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}}
)
print(resp2.choices[0].message.content)

Reasoning on
Reasoning: The user wants a haiku about GPUs. A haiku is 3 lines, 5-7-5 syllable count. Should be about GPUs. So maybe:

"Silicon minds blaze,
Parallel rivers compute fast,
Heat sings in circuits."

Check syllable counts:

Line1: "Silicon minds blaze" -> Silicon (3) minds (1) blaze (1) =5? Actually "Sil-i-con" =3, "minds"=1, "blaze"=1 =>5. Good.

Line2: "Parallel rivers compute fast" -> Parallel (3), rivers (2), compute (2), fast (1) =8? Let's count: Par-al-lel (3) maybe? Actually "parallel" is 3 syllables (par-al-lel). "rivers" 2, "com-pute" 2, "fast" 1 => total 8. Too many, need 7.

We need 7 syllables. Let's craft: "Parallel streams compute" -> Parallel (3), streams (1), com-pute (2) =6. Need 7. Add "fast": "Parallel streams compute fast" -> Parallel (3), streams (1), compute (2), fast (1) =7. Good.

Line3: "Heat sings in circuits" -> Heat (1) sings (1) in (1) cir-cuits (2) =5. Good.

So haiku:

Silicon minds blaze
Parallel streams compute fast
Heat sings in circuits

C

In [11]:
# Streaming chat completion
stream = client.chat.completions.create(
    model="nemotron",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What are the first 5 prime numbers?"}
    ],
    temperature=0.7,
    max_tokens=1024,
    stream=True,
)
for chunk in stream:
    delta = chunk.choices[0].delta
    if delta and delta.content:
        print(delta.content, end="", flush=True)


The first 5 prime numbers are: 2, 3, 5, 7, 11.

### Tool calling

Call functions using the OpenAI Tools schema and inspect returned tool_calls.

In [12]:
# Tool calling via OpenAI tools schema
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate_tip",
            "parameters": {
                "type": "object",
                "properties": {
                    "bill_total": {
                        "type": "integer",
                        "description": "The total amount of the bill"
                    },
                    "tip_percentage": {
                        "type": "integer",
                        "description": "The percentage of tip to be applied"
                    }
                },
                "required": ["bill_total", "tip_percentage"]
            }
        }
    }
]

completion = client.chat.completions.create(
    model="nemotron",
    messages=[
        {"role": "system", "content": ""},
        {"role": "user", "content": "My bill is $50. What will be the amount for 15% tip?"}
    ],
    tools=TOOLS,
    temperature=0.6,
    top_p=0.95,
    max_tokens=512,
    stream=False
)

print(completion.choices[0].message.reasoning_content)
print(completion.choices[0].message.tool_calls)

Okay, let's see. The user has a bill of $50 and wants to know the amount for a 15% tip. I need to calculate that. The tool provided is calculate_tip, which requires bill_total and tip_percentage.

First, I should check if both parameters are provided. The bill is $50, so bill_total is 50. The tip percentage is 15, so tip_percentage is 15. Both are integers, which matches the tool's requirements. 

No missing information here. So I can directly call the calculate_tip function with these values. There's no need to ask the user for more details. Let me make sure the parameters are correctly formatted as integers. Yep, 50 and 15 are both integers. 

I'll structure the tool call accordingly.

[ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-b69a86a8ac7b34c2', function=Function(arguments='{"bill_total": 50, "tip_percentage": 15}', name='calculate_tip'), type='function')]


### Video understanding

Nemotron-3-Nano-Omni is a multimodal model that can reason over text, images, audio, and video.

The cell below demonstrates video understanding: it sends a short NVIDIA Omniverse clip to the model via the OpenAI-compatible API and asks it to describe the scene.

In [2]:
video_url = "https://blogs.nvidia.com/wp-content/uploads/2023/04/nvidia-studio-itns-wk53-scene-in-omniverse-1280w.mp4"

resp = client.chat.completions.create(
    model="nemotron",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe what happens in this video."},
                {"type": "video_url", "video_url": {"url": video_url}},
            ],
        }
    ],
    temperature=0.6,
    max_tokens=2048,
)

print(resp.choices[0].message.content)


The video shows a screen recording of a 3D modeling software interface, likely Blender, displaying a winter scene. Initially, the camera presents a wide view of a small, circular island covered in snow, featuring a rustic wooden cabin nestled among tall pine trees. As the video progresses, the camera zooms in and rotates slightly toward the cabin. This closer view reveals a character dressed in furs standing near a wooden table with a cauldron on it. Surrounding the cabin are various props, including a wooden fence, a barrel, and piles of logs. The right side of the screen consistently shows the software's interface, with panels for objects, properties, and scene hierarchy.


### Controlling Reasoning Budget

The `reasoning_budget` parameter allows you to limit the length of the model's reasoning trace. When the reasoning output reaches the specified token budget, the model will attempt to gracefully end the reasoning at the next newline character. 

If no newline is encountered within 500 tokens after reaching the budget threshold, the reasoning trace will be forcibly terminated at `reasoning_budget + 500` tokens to prevent excessive generation.


In [13]:
from typing import Any, Dict, List
import openai
from transformers import AutoTokenizer


class ThinkingBudgetClient:
    def __init__(self, base_url: str, api_key: str, tokenizer_name_or_path: str):
        self.base_url = base_url
        self.api_key = api_key
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name_or_path)
        self.client = openai.OpenAI(base_url=self.base_url, api_key=self.api_key)

    def chat_completion(
        self,
        model: str,
        messages: List[Dict[str, Any]],
        reasoning_budget: int = 512,
        max_tokens: int = 1024,
        **kwargs,
    ) -> Dict[str, Any]:
        assert (
            max_tokens > reasoning_budget
        ), f"reasoning_budget must be smaller than max_tokens. Given {max_tokens=} and {reasoning_budget=}"

        # 1. first call chat completion to get reasoning content
        response = self.client.chat.completions.create(
            model=model, 
            messages=messages, 
            max_tokens=reasoning_budget, 
            **kwargs
        )
    
        reasoning_content = response.choices[0].message.reasoning_content or ""
        
        if "</think>" not in reasoning_content:
            # reasoning content is too long, closed with a period (.)
            reasoning_content = f"{reasoning_content}.\n</think>\n\n"
        
        reasoning_tokens_used = len(
            self.tokenizer.encode(reasoning_content, add_special_tokens=False)
        )
        remaining_tokens = max_tokens - reasoning_tokens_used
        
        assert (
            remaining_tokens > 0
        ), f"remaining tokens must be positive. Given {remaining_tokens=}. Increase max_tokens or lower reasoning_budget."

        # 2. append reasoning content to messages and call completion
        messages.append({"role": "assistant", "content": reasoning_content})
        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            continue_final_message=True,
        )
        
        response = self.client.completions.create(
            model=model, 
            prompt=prompt, 
            max_tokens=remaining_tokens, 
            **kwargs
        )

        response_data = {
            "reasoning_content": reasoning_content.strip().strip("</think>").strip(),
            "content": response.choices[0].text,
            "finish_reason": response.choices[0].finish_reason,
        }
        return response_data

In [14]:
# Client
client = ThinkingBudgetClient(
    base_url="http://127.0.0.1:5000/v1",
    api_key="null",
    tokenizer_name_or_path="nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16"
)

In [15]:
resp = client.chat_completion(
    model="nemotron",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1,
    max_tokens=256,
    reasoning_budget=32
)
print("Reasoning:", resp["reasoning_content"], "\nContent:", resp["content"])

Reasoning: Okay, user wants a haiku about GPUs. Hmm, they probably want something concise but meaningful since haikus are short. 

First, I should. 
Content: 
GPU whirs, light ignites,
Crunching pixels, swift and deep,
Dreams in parallel.


### Controlling Reasoning Budget Via Logit Processors

An alternative to budget control shown above it to use VLLM's logit processor functionality. This method allows the client to avoid calling the server twice. 

To use this method you first need to install `custom_logit_processors` provided in this repository under `./tools/budget`.


In [ ]:
!cd ./tools/budget && pip install -e .

Once installed, you can launch a VLLM server and point the `custom_logit_processors` via the commands:

In [ ]:
%env THINKING_BUDGET_LOGITS_PROCESSOR_ARGS={"thinking_budget":150,"thinking_budget_grace_period":30,"end_token_ids":[1338,13],"end_think_ids":[[13]],"prompt_think_ids":[12,1010]}


This sets up default arguments to the logit processor. In the above example: 

`thinking_budget` is the number of tokens the model can use in thinking/reasoning stage. 
`thinking_budget_grace_period` extra number of tokens after the budget to find a newline to gracefully stop thinking 
`end_token_ids` the sequence of tokens to artificially insert into the token stream to end thinking
`end_think_ids` the id for </think> (always 13 for this model) 
`prompt_think_ids` the sequence to ids to allow the logit processor to recognize that the model is in thinking stage (always [12, 1010] for this model) 

Now we can launch the VLLM server.

In [ ]:
!python3 -m vllm.entrypoints.openai.api_server \
  --served-model-name "model" \
  --model nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16 \
  --logits-processors "custom_logit_processors.v1.nano_v3_logit_processors:ThinkingBudgetLogitsProcessor" \
  --port 8881 \
  --trust-remote-code

Once the server is running, we can use the client to control the reasoning budget.
An example client is provided in `./tools/budget/client.py`. 

We can use the client without any additional arguments to use the default thinking budget which was set when the VLLM server was launched..


In [ ]:
client = OpenAI(
    base_url="http://localhost:8881/v1", # Your vLLM server URL
    api_key="EMPTY"
)

result = client.chat.completions.create(
    model="model",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Consider all the ways you can interpret the question 'What is 5.9 plus 6.1' and give the best answer possible."}
    ],
    temperature=1.0,
    max_tokens=12200, # uses the default thinking budget set during starting of the vllm server.
)

This will trigger the default truncation behavior around 150 tokens  + 30 grace tokens. The output of the reasoning part should look something like this:

The user wants "Consider all the ways you can interpret the question 'What is 5.9 plus 6.1' and give the best answer possible."

We need to interpret the phrase in various ways – maybe as basic arithmetic, as concatenation, as some alternative base, maybe as a trick question. Provide the best answer possible.

We need to be thorough: cover decimal addition, interpretation in base X, rounding, approximation, maybe interpretation as time (5.9 seconds + 6.1 seconds) = 12.0 seconds, maybe as different numeral systems, as measurement with units.

The question likely wants all possible interpretations and then provide the best answer.

Thus: interpret as simple addition in base 10 yields 12.0. Could also be interpreted as 5.9 + 6.1 = 12.0 exactly because 0..

</think>


The client may alter the defaults during each call as shown below:

In [ ]:
custom_think_budget = 10
custom_think_budget_grace_period = 10
custom_think_truncation = [1871, 5565, 11483, 6139, 2016, 1536, 6934, 1338, 13] # "Reached thining limit set by client\n\n</think>

result = client.chat.completions.create(
    model="model",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Consider all the ways you can interpret the question 'What is 5.9 plus 6.1' and give the best answer possible."}
    ],
    temperature=1.0,
    max_tokens=12200,
    logprobs=False,
    extra_body={
        "vllm_xargs": {
            "thinking_budget": custom_think_budget,
            "thinking_budget_grace_period": custom_think_budget_grace_period,
            "end_token_ids": json.dumps(custom_think_truncation),
        }
    }
)

Which results in a reasoning trace rather brutally truncated like this:

The user asks: "Consider all the ways you can interpret the question 'What is 5. Reached thinking limit set by client.

</think>

## Cleanup and shutdown

To free resources after this notebook:

1. Release in-process model memory (run the next cell).
2. Stop the OpenAI-compatible `vllm serve` process in the terminal where it was started (`Ctrl+C`).
3. If needed, restart the kernel to ensure a clean state.

In [16]:
import gc
import torch

# Release in-process model/tokenizer objects if present
for name in ("llm", "tokenizer"):
    if name in globals():
        del globals()[name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("In-process cleanup complete.")
print("If vllm serve is running in a terminal, stop it with Ctrl+C.")

In-process cleanup complete.
If vllm serve is running in a terminal, stop it with Ctrl+C.
